# Ecobici CDMX - Predicción de Disponibilidad con ML

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.utils.class_weight import compute_class_weight
import warnings
import json

# Desactivar alertas para asegurar una salida JSON limpia en la consola
warnings.filterwarnings('ignore')
 
df = pd.read_csv('ecobici_datos_20k.csv')

# 1. ORDENAMIENTO Y PREPARACIÓN DE DATOS
df['run_ts'] = pd.to_datetime(df['run_ts'], errors='coerce')
df = df.dropna(subset=['run_ts'])
df = df.sort_values(by=['station_id', 'run_ts']).reset_index(drop=True)

df['pct_full_future_2h'] = df.groupby('station_id')['pct_full'].shift(-12)
df['bikes_avail_future_2h'] = df.groupby('station_id')['bikes_avail'].shift(-12)

def asignar_rango_ocupacion_estricto(pct):
    if pd.isna(pct): return np.nan
    if pct <= 0.30: return 0    
    elif pct <= 0.55: return 1  
    elif pct <= 0.85: return 2  
    else: return 3              

df['target_rango_2h'] = df['pct_full_future_2h'].apply(asignar_rango_ocupacion_estricto)

# 2. INGENIERÍA DE VARIABLES
df['hora'] = df['run_ts'].dt.hour
df['dia_semana'] = df['run_ts'].dt.dayofweek
df['es_fin_de_semana'] = df['dia_semana'].apply(lambda x: 1 if x >= 5 else 0)
df['hora_sin'] = np.sin(2 * np.pi * df['hora'] / 24.0)
df['hora_cos'] = np.cos(2 * np.pi * df['hora'] / 24.0)
df['hora_tipo_dia'] = df['hora'] + (df['es_fin_de_semana'] * 24)

df['pct_full_log1'] = df.groupby('station_id')['pct_full'].shift(1)
df['pct_full_log2'] = df.groupby('station_id')['pct_full'].shift(2)
df['pct_full_log3'] = df.groupby('station_id')['pct_full'].shift(3)
df['pct_full_log_1h'] = df.groupby('station_id')['pct_full'].shift(6)
df['pct_full_log_2h'] = df.groupby('station_id')['pct_full'].shift(12)

df['velocidad_cambio'] = df['pct_full_log1'] - df['pct_full_log2']
df['aceleracion_cambio'] = df['pct_full_log1'] - df['pct_full_log3']
df['tendencia_1h'] = df['pct_full_log1'] - df['pct_full_log_1h']
df['aceleracion_tendencia_1h'] = df['velocidad_cambio'] - (df['pct_full_log_1h'] - df['pct_full_log_2h'])

if 'ocupacion_historica_estacion_hora' not in df.columns:
    mapa_historico = df.groupby(['station_id', 'hora'])['pct_full'].mean().reset_index()
    mapa_historico.rename(columns={'pct_full': 'ocupacion_historica_estacion_hora'}, inplace=True)
    df = df.merge(mapa_historico, on=['station_id', 'hora'], how='left')

if 'zona_logistica' in df.columns: df = df.drop(columns=['zona_logistica'])
coords = df[['lat', 'lon']].drop_duplicates().reset_index(drop=True)
kmeans_final = KMeans(n_clusters=5, random_state=42, n_init=10)
coords['zona_logistica'] = kmeans_final.fit_predict(coords[['lat', 'lon']])
df = df.merge(coords, on=['lat', 'lon'], how='left')

if 'rango_historico_estacion' in df.columns: df = df.drop(columns=['rango_historico_estacion'])
encoding_map = df.groupby(['station_id', 'hora_tipo_dia'])['target_rango_2h'].mean().reset_index()
encoding_map.rename(columns={'target_rango_2h': 'rango_historico_estacion'}, inplace=True)
df = df.merge(encoding_map, on=['station_id', 'hora_tipo_dia'], how='left')

# 3. ENTRENAMIENTO DE MODELO HÍBRIDO
columnas_limpieza = ['target_rango_2h', 'bikes_avail_future_2h', 'pct_full_log1', 'pct_full_log2', 'rango_historico_estacion']
df_xgb = df.dropna(subset=columnas_limpieza)
df_xgb = df_xgb[(df_xgb['is_installed'] == 1) & (df_xgb['is_renting'] == 1) & (df_xgb['is_returning'] == 1)]

df_xgb = df_xgb.sort_values('run_ts')
split_idx = int(len(df_xgb) * 0.8)
train_df = df_xgb.iloc[:split_idx].copy()
test_df = df_xgb.iloc[split_idx:].copy()

features_regresor = [
    'capacity', 'bikes_avail', 'pct_full', 'hora_sin', 'hora_cos', 'dia_semana', 
    'es_fin_de_semana', 'hora_tipo_dia', 'pct_full_log1', 'pct_full_log2', 'pct_full_log3',
    'pct_full_log_1h', 'pct_full_log_2h', 'rango_historico_estacion', 'velocidad_cambio', 
    'aceleracion_cambio', 'tendencia_1h', 'aceleracion_tendencia_1h', 'ocupacion_historica_estacion_hora', 'zona_logistica'
]

model_regresor = xgb.XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=5, objective='reg:squarederror', random_state=42)
model_regresor.fit(train_df[features_regresor], train_df['bikes_avail_future_2h'])

train_df['pred_num_regresor'] = model_regresor.predict(train_df[features_regresor])
test_df['pred_num_regresor'] = model_regresor.predict(test_df[features_regresor])

features_clasificador = features_regresor + ['pred_num_regresor']
y_train_cl = train_df['target_rango_2h'].astype(int)
pesos_clase = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_cl), y=y_train_cl)
pesos_muestras_train = y_train_cl.map(lambda x: pesos_clase[x])

model_clasificador = xgb.XGBClassifier(n_estimators=400, learning_rate=0.03, max_depth=5, objective='multi:softprob', num_class=4, random_state=42)
model_clasificador.fit(train_df[features_clasificador], y_train_cl, sample_weight=pesos_muestras_train)

test_df['estado_predicho'] = model_clasificador.predict(test_df[features_clasificador])
test_df['bicis_predichas_2h'] = np.round(test_df['pred_num_regresor'])
test_df['inventario_optimo'] = np.round(test_df['capacity'] * 0.425)

def calcular_unidades_netas(row):
    if row['estado_predicho'] == 0:  
        return int(max(1, row['inventario_optimo'] - row['bicis_predichas_2h']))
    elif row['estado_predicho'] == 3:  
        return int(-max(1, row['bicis_predichas_2h'] - row['inventario_optimo']))
    return 0

test_df['unidades_requeridas'] = test_df.apply(calcular_unidades_netas, axis=1)

# 4. ALGORITMO DE RUTEO GEOGRÁFICO CON DISTANCIA DIRECTA
def calcular_distancia_directa(lat1, lon1, lat2, lon2):
    # Suma directa del valor absoluto de las diferencias 
    return abs(lat1 - lat2) + abs(lon1 - lon2)

def asignar_vehiculo(cantidad):
    if cantidad > 32:
        return "Camión Grande (Cap 50)"
    elif cantidad > 16:
        return "Camioneta + Remolque (Cap 32)"
    else:
        return "Camioneta Ligera (Cap 16 - Sin Remolque)"

ts_reciente = test_df['run_ts'].max()
df_despacho_instante = test_df[test_df['unidades_requeridas'] != 0].copy()

rutas_generadas = []
distancia_limite_grados = 0.009 # 0.009 grados coordenados equivalen aproximadamente a un radio de 1.0 km en la realidad

for zona, grupo in df_despacho_instante.groupby('zona_logistica'):
    estaciones_oferta = grupo[grupo['unidades_requeridas'] < 0].copy()
    estaciones_demanda = grupo[grupo['unidades_requeridas'] > 0].copy()
    
    oferta_list = estaciones_oferta[['station_id', 'lat', 'lon', 'unidades_requeridas']].to_dict('records')
    demanda_list = estaciones_demanda[['station_id', 'lat', 'lon', 'unidades_requeridas']].to_dict('records')
    
    for o in oferta_list: o['disponibles'] = abs(o['unidades_requeridas'])
    for d in demanda_list: d['necesitadas'] = d['unidades_requeridas']
    
    # PASO A: Emparejamiento por Circuito Cerrado Local (Filtrado por Proximidad Directa)
    for d in demanda_list:
        while d['necesitadas'] > 0:
            ofertas_validas = [o for o in oferta_list if int(o['station_id']) != int(d['station_id']) and o['disponibles'] > 0]
            
            if not ofertas_validas:
                break 
                
            # Calcular distancias directas en grados planos
            for o in ofertas_validas:
                o['distancia'] = calcular_distancia_directa(d['lat'], d['lon'], o['lat'], o['lon'])
            
            # FILTRAR: Conservar únicamente las estaciones dentro del radio equivalente en grados
            ofertas_dentro_del_radio = [o for o in ofertas_validas if o['distancia'] <= distancia_limite_grados]
            
            if not ofertas_dentro_del_radio:
                break 
            
            # Ordenar por cercanía de coordenadas
            ofertas_dentro_del_radio = sorted(ofertas_dentro_del_radio, key=lambda x: x['distancia'])
            mas_cercana = ofertas_dentro_del_radio[0]
            
            cantidad_a_transferir = min(d['necesitadas'], mas_cercana['disponibles'])
            
            if cantidad_a_transferir > 0:
                distancia_estimada_km = mas_cercana['distancia'] * 111.0
                
                rutas_generadas.append({
                    'Zona Logística': int(zona),
                    'Estación Origen (VACIAR)': int(mas_cercana['station_id']),
                    'Estación Destino (LLENAR)': int(d['station_id']),
                    'Bicicletas a Mover': int(cantidad_a_transferir),
                    'Distancia (Km)': round(distancia_estimada_km, 2),
                    'Vehículo Asignado': asignar_vehiculo(cantidad_a_transferir)
                })
                
                d['necesitadas'] -= cantidad_a_transferir
                
                for o in oferta_list:
                    if o['station_id'] == mas_cercana['station_id']:
                        o['disponibles'] -= cantidad_a_transferir

    # PASO B: Captura de Estaciones de Demanda Huérfanas
    for d in demanda_list:
        if d['necesitadas'] > 0:
            rutas_generadas.append({
                'Zona Logística': int(zona),
                'Estación Origen (VACIAR)': "Almacén Central",
                'Estación Destino (LLENAR)': int(d['station_id']),
                'Bicicletas a Mover': int(d['necesitadas']),
                'Distancia (Km)': None,
                'Vehículo Asignado': asignar_vehiculo(d['necesitadas'])
            })

    # PASO C: Captura de Estaciones de Oferta Huérfanas
    for o in oferta_list:
        if o['disponibles'] > 0:
            rutas_generadas.append({
                'Zona Logística': int(zona),
                'Estación Origen (VACIAR)': int(o['station_id']),
                'Estación Destino (LLENAR)': "Almacén Central",
                'Bicicletas a Mover': int(o['disponibles']),
                'Distancia (Km)': None,
                'Vehículo Asignado': asignar_vehiculo(o['disponibles'])
            })

df_rutas = pd.DataFrame(rutas_generadas)

# 5. CÁLCULO DE MÉTRICAS OPERATIVAS
acc_semaforo = accuracy_score(test_df['target_rango_2h'].astype(int), test_df['estado_predicho'].astype(int))
mae_volumen = mean_absolute_error(test_df['bikes_avail_future_2h'], test_df['pred_num_regresor'])

df_criticas = test_df[test_df['estado_predicho'].isin([0, 3])].copy()
total_compensado = 0
total_necesitado = 0

for zona, grupo in df_criticas.groupby('zona_logistica'):
    demand_llenar = grupo[grupo['unidades_requeridas'] > 0]['unidades_requeridas'].sum()
    supply_vaciar = abs(grupo[grupo['unidades_requeridas'] < 0]['unidades_requeridas'].sum())
    total_compensado += min(demand_llenar, supply_vaciar)
    total_necesitado += demand_llenar

# 6. CONVERSIÓN A JSON DE API
def generar_payload_api(df_rutas, acc_semaforo, mae_volumen, total_compensado, total_necesitado):
    distancia_local = df_rutas['Distancia (Km)'].dropna().sum() if not df_rutas.empty else 0.0

    metricas = {
        "model_performance": {
            "accuracy_semaforo_pct": round(acc_semaforo * 100, 2),
            "mae_volumen_bicicletas": round(mae_volumen, 2)
        },
        "green_logistics": {
            "movimientos_mitigados_unidades": int(total_compensado),
            "eficiencia_rebalanceo_local_pct": round((total_compensado / total_necesitado * 100), 2) if total_necesitado > 0 else 0.0,
            "distancia_total_optimizada_local_km": round(distancia_local, 2)
        },
        "flota_resumen": df_rutas['Vehículo Asignado'].value_counts().to_dict() if not df_rutas.empty else {}
    }
    
    rutas_list = []
    if not df_rutas.empty:
        df_api = df_rutas.rename(columns={
            'Zona Logística': 'zonaLogistica',
            'Estación Origen (VACIAR)': 'estacionOrigen',
            'Estación Destino (LLENAR)': 'estacionDestino',
            'Bicicletas a Mover': 'bicicletasAMover',
            'Distancia (Km)': 'distanciaKm',
            'Vehículo Asignado': 'vehiculoAsignado'
        })
        df_api['distanciaKm'] = df_api['distanciaKm'].replace({np.nan: None})
        rutas_list = df_api.to_dict(orient='records')
        
    api_output = {
        "status": "success",
        "timestamp_evaluacion": str(ts_reciente),
        "metricas_globales": metricas,
        "hoja_de_ruta": rutas_list
    }
    
    return json.dumps(api_output, indent=4, ensure_ascii=False)

json_para_backend = generar_payload_api(df_rutas, acc_semaforo, mae_volumen, total_compensado, total_necesitado)
print(json_para_backend)

{
    "status": "success",
    "timestamp_evaluacion": "2025-09-20 18:18:57-06:00",
    "metricas_globales": {
        "model_performance": {
            "accuracy_semaforo_pct": 74.64,
            "mae_volumen_bicicletas": 2.88
        },
        "green_logistics": {
            "movimientos_mitigados_unidades": 1522,
            "eficiencia_rebalanceo_local_pct": 11.18,
            "distancia_total_optimizada_local_km": 55.66
        },
        "flota_resumen": {
            "Camioneta Ligera (Cap 16 - Sin Remolque)": 1647
        }
    },
    "hoja_de_ruta": [
        {
            "zonaLogistica": 0,
            "estacionOrigen": 224,
            "estacionDestino": 280,
            "bicicletasAMover": 4,
            "distanciaKm": 0.32,
            "vehiculoAsignado": "Camioneta Ligera (Cap 16 - Sin Remolque)"
        },
        {
            "zonaLogistica": 0,
            "estacionOrigen": 280,
            "estacionDestino": 224,
            "bicicletasAMover": 1,
            "di